# Audit và loại pack SFT chứa sample bị truncate

Notebook này chạy hoàn toàn local, dành cho output của Nemotron Lightning 3.5 SFT data prep.

Notebook thực hiện:

1. Đọc nhanh statistics từ receipt trong `runs/` tương ứng với các shard hiện có trong `splits/`.
2. Phân biệt `num_truncated` (do `max_doc_tokens`) và `num_truncated_to_pack_size` (do `pack_size`).
3. Đọc `spool/lengths.bin`, dựng lại đúng bin assignment bằng `algorithm` và `seed`.
4. Xác định chính xác packed row chứa sample có chiều dài gốc `> pack_size`.
5. Ghi một folder splits mới, bỏ toàn bộ pack chứa sample đó; không sửa dữ liệu gốc.
6. Thống kê zero-loss pack, packing utilization và kiểm tra output.

> Giới hạn: sample đã bị cắt bởi `max_doc_tokens` không thể map ngược về packed row vì spool chỉ lưu chiều dài **sau** bước cắt này. Mặc định notebook sẽ không cho tạo output 'clean' nếu phát hiện loại truncation đó.

In [ ]:
from pathlib import Path

# ===== Chỉ cần sửa cell này =====
OUTPUT_ROOT = Path("/path/to/output/stage1_sft")
SPLITS_DIR = OUTPUT_ROOT / "splits"
CLEAN_SPLITS_DIR = OUTPUT_ROOT / "splits_without_truncated"
PACK_SIZE = 4096

# False: chỉ audit/preview. Đổi thành True khi đã kiểm tra statistics.
WRITE_CLEANED_SPLITS = False

# Khuyến nghị giữ True: không tạo output nếu spool thiếu hoặc có max_doc_tokens truncation.
REQUIRE_SPOOL = True
FAIL_IF_MAX_DOC_TRUNCATED = True

PARQUET_BATCH_SIZE = 1024
PARQUET_COMPRESSION = "zstd"

In [ ]:
import json
import os
import random
from array import array
from collections import defaultdict
from dataclasses import asdict, dataclass
from typing import Any

import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None


def show_rows(rows):
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        display(rows)


assert SPLITS_DIR.is_dir(), f"Không tìm thấy splits folder: {SPLITS_DIR}"
assert PACK_SIZE > 0
print(f"Input splits : {SPLITS_DIR}")
print(f"Clean output : {CLEAN_SPLITS_DIR}")
print(f"Pack size    : {PACK_SIZE}")

## 1. Khám phá shard và đọc nhanh statistics trong `runs`

Các file trong `splits/` thường là symlink đến:

```text
runs/<run_hash>/datasets/<dataset>/<plan_hash>/shard_N.parquet
```

Receipt tương ứng nằm tại `receipts/shard_N.json`. Chỉ receipt của shard đang được tham chiếu bởi `splits/` được tính, tránh cộng nhầm các run cũ.

In [ ]:
def discover_split_shards(splits_dir: Path) -> dict[str, list[Path]]:
    result = {}
    for split_dir in sorted(path for path in splits_dir.iterdir() if path.is_dir()):
        shards = sorted(path for path in split_dir.glob("*.parquet") if path.is_file())
        if shards:
            result[split_dir.name] = shards
    if not result:
        raise FileNotFoundError(f"Không tìm thấy *.parquet dưới {splits_dir}")
    return result


def receipt_path_for_shard(split_shard: Path) -> Path:
    source = split_shard.resolve()
    return source.parent / "receipts" / f"{source.stem}.json"


def spool_dir_for_shard(split_shard: Path) -> Path:
    source = split_shard.resolve()
    return source.parent / "spool" / source.stem


split_shards = discover_split_shards(SPLITS_DIR)
receipt_rows = []
receipt_by_shard = {}

for split, shards in split_shards.items():
    for shard in shards:
        receipt_path = receipt_path_for_shard(shard)
        if not receipt_path.is_file():
            raise FileNotFoundError(f"Thiếu receipt cho {shard}: {receipt_path}")
        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        if receipt.get("status") != "completed":
            raise RuntimeError(f"Receipt chưa completed: {receipt_path}")
        stats = receipt.get("stats", {}) or {}
        receipt_by_shard[shard] = receipt
        receipt_rows.append({
            "split": split,
            "shard": shard.name,
            "dataset": receipt.get("dataset_name"),
            "num_sequences": int(stats.get("num_sequences", 0) or 0),
            "num_packs": int(stats.get("num_packed_sequences", 0) or 0),
            "tokens": int(stats.get("total_tokens", 0) or 0),
            "truncated_max_doc": int(stats.get("num_truncated", 0) or 0),
            "truncated_pack_size": int(stats.get("num_truncated_to_pack_size", 0) or 0),
            "filtered": int(stats.get("num_filtered", 0) or 0),
            "empty_after_masking": int(stats.get("num_empty_after_masking", 0) or 0),
            "receipt": str(receipt_path),
        })

print(f"Đã đọc {len(receipt_rows)} completed receipts.")
show_rows(receipt_rows[:20])

In [ ]:
def aggregate_receipts(rows):
    grouped = defaultdict(lambda: defaultdict(int))
    numeric_fields = [
        "num_sequences", "num_packs", "tokens",
        "truncated_max_doc", "truncated_pack_size",
        "filtered", "empty_after_masking",
    ]
    for row in rows:
        for key in numeric_fields:
            grouped[row["split"]][key] += row[key]
            grouped["TOTAL"][key] += row[key]
        grouped[row["split"]]["files"] += 1
        grouped["TOTAL"]["files"] += 1
    result = []
    for split in [*sorted(key for key in grouped if key != "TOTAL"), "TOTAL"]:
        item = {"split": split, **grouped[split]}
        item["pack_truncation_rate"] = (
            item["truncated_pack_size"] / item["num_sequences"]
            if item["num_sequences"] else 0.0
        )
        result.append(item)
    return result


receipt_summary = aggregate_receipts(receipt_rows)
show_rows(receipt_summary)

total_receipt = next(row for row in receipt_summary if row["split"] == "TOTAL")
print(f"Pack-size truncations : {total_receipt['truncated_pack_size']:,}")
print(f"Max-doc truncations   : {total_receipt['truncated_max_doc']:,}")

## 2. Dựng lại packing và xác định chính xác pack cần bỏ

`lengths.bin` chứa chiều dài sequence trước khi clamp bởi `pack_size`. Notebook chạy lại đúng thuật toán packing từ manifest và đánh dấu bin chứa ít nhất một sequence có `original_length > pack_size`.

Số pack tìm được sẽ được cross-check với `num_truncated_to_pack_size` trong receipt.

In [ ]:
def read_uint32_file(path: Path) -> list[int]:
    values = array("I")
    if values.itemsize != 4:
        raise RuntimeError(f"Platform unsigned-int size={values.itemsize}, expected 4")
    size = path.stat().st_size
    if size % 4:
        raise ValueError(f"Invalid uint32 file size: {path} ({size} bytes)")
    with path.open("rb") as stream:
        values.fromfile(stream, size // 4)
    return [int(value) for value in values]


def reconstruct_bins(lengths: list[int], pack_size: int, algorithm: str, seed: int | None):
    indexed = [(min(length, pack_size), index) for index, length in enumerate(lengths)]

    if algorithm == "concatenative":
        bins, current_size = [[]], 0
        for length, index in indexed:
            if current_size + length <= pack_size:
                bins[-1].append(index)
                current_size += length
            else:
                bins.append([index])
                current_size = length
        return [item for item in bins if item]

    if algorithm == "first_fit_decreasing":
        indexed.sort(key=lambda item: item[0], reverse=True)
    elif algorithm == "first_fit_shuffle":
        random.Random(seed).shuffle(indexed)
    else:
        raise ValueError(f"Unsupported packing algorithm: {algorithm}")

    bins, remaining = [], []
    for length, index in indexed:
        for bin_index, capacity in enumerate(remaining):
            if capacity >= length:
                bins[bin_index].append(index)
                remaining[bin_index] -= length
                break
        else:
            bins.append([index])
            remaining.append(pack_size - length)
    return bins


def analyze_shard(split: str, shard: Path, receipt: dict[str, Any]):
    spool_dir = spool_dir_for_shard(shard)
    manifest_path = spool_dir / "manifest.json"
    lengths_path = spool_dir / "lengths.bin"
    if not manifest_path.is_file() or not lengths_path.is_file():
        if REQUIRE_SPOOL:
            raise FileNotFoundError(f"Thiếu spool cho {shard}: {spool_dir}")
        return None

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_pack_size = int(manifest.get("pack_size", 0) or 0)
    if manifest_pack_size != PACK_SIZE:
        raise ValueError(f"pack_size mismatch tại {manifest_path}: {manifest_pack_size} != {PACK_SIZE}")

    lengths = read_uint32_file(lengths_path)
    bins = reconstruct_bins(
        lengths, PACK_SIZE, str(manifest.get("algorithm")),
        None if manifest.get("seed") is None else int(manifest["seed"]),
    )
    parquet_rows = pq.ParquetFile(shard).metadata.num_rows
    if len(bins) != parquet_rows:
        raise ValueError(f"Bin/Parquet mismatch {shard}: {len(bins)} != {parquet_rows}")

    candidate_rows = []
    candidate_details = []
    for pack_row, sequence_indices in enumerate(bins):
        truncated = [index for index in sequence_indices if lengths[index] > PACK_SIZE]
        if truncated:
            candidate_rows.append(pack_row)
            candidate_details.append({
                "split": split,
                "shard": shard.name,
                "pack_row": pack_row,
                "sequence_indices": sequence_indices,
                "truncated_sequence_indices": truncated,
                "original_lengths": [lengths[index] for index in truncated],
                "tokens_removed": sum(lengths[index] - PACK_SIZE for index in truncated),
            })

    receipt_stats = receipt.get("stats", {}) or {}
    expected = int(receipt_stats.get("num_truncated_to_pack_size", 0) or 0)
    if len(candidate_rows) != expected:
        raise ValueError(
            f"Cross-check failed {shard}: spool tìm {len(candidate_rows)}, receipt báo {expected}"
        )
    return {
        "candidate_rows": set(candidate_rows),
        "candidate_details": candidate_details,
        "num_sequences": len(lengths),
        "num_packs": len(bins),
        "algorithm": manifest.get("algorithm"),
        "seed": manifest.get("seed"),
        "max_doc_tokens": manifest.get("max_doc_tokens"),
    }

In [ ]:
analysis_by_shard = {}
candidate_details = []

for split, shards in split_shards.items():
    for shard in shards:
        result = analyze_shard(split, shard, receipt_by_shard[shard])
        if result is None:
            raise RuntimeError(f"Không thể xác định chính xác truncation vì thiếu spool: {shard}")
        analysis_by_shard[shard] = result
        candidate_details.extend(result["candidate_details"])

detected_truncated = len(candidate_details)
receipt_truncated = total_receipt["truncated_pack_size"]
assert detected_truncated == receipt_truncated, (detected_truncated, receipt_truncated)

print(f"Truncated samples/packs xác định chính xác: {detected_truncated:,}")
print(f"Tokens bị cắt tối thiểu               : {sum(x['tokens_removed'] for x in candidate_details):,}")
show_rows(candidate_details[:50])

## 3. Scan packed Parquet để kiểm tra loss mask và utilization

Phần này chậm hơn receipt scan nhưng hữu ích để phát hiện pack không có token trainable. Nó không decode token và không cần tokenizer.

In [ ]:
def scan_packed_shard(path: Path):
    stats = {"packs": 0, "sequences": 0, "tokens": 0, "trainable_tokens": 0,
             "zero_trainable_packs": 0, "zero_trainable_sequences": 0}
    parquet = pq.ParquetFile(path)
    required = {"input_ids", "loss_mask", "seq_start_id"}
    if not required.issubset(parquet.schema_arrow.names):
        raise ValueError(f"Invalid packed schema: {path}")

    for batch in parquet.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=list(required)):
        rows = batch.to_pydict()
        for input_ids, loss_mask, starts in zip(
            rows["input_ids"], rows["loss_mask"], rows["seq_start_id"], strict=True
        ):
            if len(input_ids) != len(loss_mask):
                raise ValueError(f"input_ids/loss_mask mismatch: {path}")
            ends = starts[1:] + [len(loss_mask)]
            per_sequence = [sum(loss_mask[start:end]) for start, end in zip(starts, ends, strict=True)]
            trainable = sum(per_sequence)
            stats["packs"] += 1
            stats["sequences"] += len(starts)
            stats["tokens"] += len(input_ids)
            stats["trainable_tokens"] += trainable
            stats["zero_trainable_packs"] += int(trainable == 0)
            stats["zero_trainable_sequences"] += sum(value == 0 for value in per_sequence)
    return stats


packed_rows = []
for split, shards in split_shards.items():
    aggregate = defaultdict(int)
    for shard in shards:
        for key, value in scan_packed_shard(shard).items():
            aggregate[key] += value
    aggregate["split"] = split
    aggregate["packing_utilization"] = (
        aggregate["tokens"] / (aggregate["packs"] * PACK_SIZE) if aggregate["packs"] else 0.0
    )
    packed_rows.append(dict(aggregate))

show_rows(packed_rows)

## 4. Ghi cleaned splits

Cell dưới đây chỉ thực sự ghi file khi `WRITE_CLEANED_SPLITS = True`. Output phải chưa tồn tại hoặc là folder rỗng. Không có thao tác xóa hay overwrite dữ liệu hiện hữu.

In [ ]:
def write_filtered_parquet(source: Path, destination: Path, drop_rows: set[int]):
    parquet = pq.ParquetFile(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    writer = pq.ParquetWriter(temporary, parquet.schema_arrow, compression=PARQUET_COMPRESSION)
    row_offset = 0
    kept = dropped = 0
    try:
        for batch in parquet.iter_batches(batch_size=PARQUET_BATCH_SIZE):
            keep = [row_offset + index not in drop_rows for index in range(batch.num_rows)]
            row_offset += batch.num_rows
            dropped += keep.count(False)
            kept += keep.count(True)
            table = pa.Table.from_batches([batch]).filter(pa.array(keep, type=pa.bool_()))
            if table.num_rows:
                writer.write_table(table)
    except Exception:
        writer.close()
        temporary.unlink(missing_ok=True)
        raise
    else:
        writer.close()
        os.replace(temporary, destination)
    return {"kept": kept, "dropped": dropped}


if not WRITE_CLEANED_SPLITS:
    print("DRY RUN: chưa ghi dữ liệu. Đổi WRITE_CLEANED_SPLITS=True rồi chạy lại cell này.")
else:
    max_doc_count = total_receipt["truncated_max_doc"]
    if FAIL_IF_MAX_DOC_TRUNCATED and max_doc_count:
        raise RuntimeError(
            f"Có {max_doc_count} sequence đã bị max_doc_tokens truncate và không thể map ngược chính xác. "
            "Hãy lọc/rebuild từ raw data trước khi tạo cleaned splits."
        )
    if CLEAN_SPLITS_DIR.resolve() == SPLITS_DIR.resolve():
        raise ValueError("CLEAN_SPLITS_DIR không được trùng SPLITS_DIR")
    if CLEAN_SPLITS_DIR.exists() and any(CLEAN_SPLITS_DIR.iterdir()):
        raise FileExistsError(f"Output phải chưa tồn tại hoặc rỗng: {CLEAN_SPLITS_DIR}")

    write_rows = []
    for split, shards in split_shards.items():
        for shard in shards:
            destination = CLEAN_SPLITS_DIR / split / shard.name
            result = write_filtered_parquet(
                shard, destination, analysis_by_shard[shard]["candidate_rows"]
            )
            write_rows.append({"split": split, "shard": shard.name, **result})

    report = {
        "source_splits": str(SPLITS_DIR),
        "clean_splits": str(CLEAN_SPLITS_DIR),
        "pack_size": PACK_SIZE,
        "receipt_summary": receipt_summary,
        "truncated_pack_details": candidate_details,
        "files": write_rows,
        "total_packs_dropped": sum(row["dropped"] for row in write_rows),
        "total_packs_kept": sum(row["kept"] for row in write_rows),
    }
    report_path = CLEAN_SPLITS_DIR / "truncation_filter_report.json"
    report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"Đã ghi cleaned splits: {CLEAN_SPLITS_DIR}")
    print(f"Đã bỏ {report['total_packs_dropped']:,} pack")
    print(f"Report: {report_path}")
    show_rows(write_rows[:20])

## 5. Verify cleaned output

Chạy sau khi đã bật `WRITE_CLEANED_SPLITS`. Cell kiểm tra số row output đúng bằng input trừ số pack bị loại và schema không thay đổi.

In [ ]:
if not WRITE_CLEANED_SPLITS:
    print("Bỏ qua verify vì đang ở DRY RUN.")
else:
    verify_rows = []
    for split, shards in split_shards.items():
        for source in shards:
            cleaned = CLEAN_SPLITS_DIR / split / source.name
            source_file = pq.ParquetFile(source)
            cleaned_file = pq.ParquetFile(cleaned)
            expected = source_file.metadata.num_rows - len(analysis_by_shard[source]["candidate_rows"])
            actual = cleaned_file.metadata.num_rows
            if actual != expected:
                raise ValueError(f"Row count mismatch {cleaned}: {actual} != {expected}")
            if cleaned_file.schema_arrow != source_file.schema_arrow:
                raise ValueError(f"Schema mismatch: {cleaned}")
            verify_rows.append({"split": split, "shard": source.name, "rows": actual, "status": "OK"})

    print(f"Verify thành công {len(verify_rows)} shards.")
    show_rows(verify_rows[:20])
    print("\nTraining paths:")
    for split in sorted(split_shards):
        print(f"  {split}: {CLEAN_SPLITS_DIR / split}/")

## Diễn giải kết quả

- `truncated_pack_size`: số sequence có chiều dài trước packing lớn hơn `pack_size`. Với packer hiện tại, mỗi sequence này chiếm một pack riêng.
- `truncated_max_doc`: số sequence đã bị cắt sớm bởi `max_doc_tokens`; không thể xác định packed row từ spool hiện tại.
- `zero_trainable_packs`: pack có tổng `loss_mask == 0`; nên kiểm tra riêng vì chúng không tạo gradient hữu ích. Notebook chỉ thống kê, không tự loại nhóm này.
- `packing_utilization`: `tokens / (num_packs * pack_size)`.

Nếu `truncated_max_doc > 0`, cách chính xác nhất là quay lại raw conversations, render bằng đúng tokenizer/chat template, rồi lọc hoặc truncate theo turn trước khi chạy data prep lại.